# 09 · End to end: training and evaluating a neural network

**Objective:** To put all the pieces together on a real dataset: split the data, preprocess it without information leaking from the test set, train a neural network with `DataLoader`s, monitor it on a validation set, compare it with baselines, evaluate it once on the test set, and save it so that the result can be reproduced.

**Dataset:** the *digits* dataset: 1,797 grayscale images of handwritten digits, 8×8 pixels each, 10 classes. It ships with scikit-learn, so no download is needed.

## Part 1: Concepts & Guided Coding (90 mins)

### 0. Setup and reproducibility
Neural network training is random: initial weights, shuffling of the batches, dropout. To be able to rerun an experiment and get the same numbers, fix the random seeds of every library involved, and record the library versions.
Even so, results can differ across hardware (CPU vs GPU) and library versions; on GPU, `torch.use_deterministic_algorithms(True)` enforces deterministic operations at some cost in speed.

In [ ]:
import copy
import json
import platform
import random

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

SEED = 0
set_seed(SEED)

versions = {"python": platform.python_version(), "numpy": np.__version__,
            "torch": torch.__version__, "scikit-learn": sklearn.__version__}
print(versions)

### 1. Load and inspect the data

In [ ]:
digits = load_digits()
X = digits.data.astype(np.float32)   # shape (1797, 64): each 8x8 image flattened into 64 features
y = digits.target                    # integer labels 0..9

print("X:", X.shape, X.dtype, " pixel range:", X.min(), "-", X.max())
print("y:", y.shape, " samples per class:", np.bincount(y))

fig, axes = plt.subplots(2, 8, figsize=(10, 3))
for ax, image, label in zip(axes.flat, digits.images, digits.target):
    ax.imshow(image, cmap="gray_r")
    ax.set_title(label)
    ax.axis("off")
plt.show()

### 2. Split: training, validation and test sets
- **Training set:** used to fit the parameters (weights).
- **Validation set:** used to make choices: number of epochs, architecture, learning rate...
- **Test set:** used **once**, at the end, to estimate performance on new data. If we use it to make choices, the estimate becomes optimistic.

We use 60% / 20% / 20%. `stratify` keeps the class proportions the same in the three sets.

In [ ]:
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=SEED)

print(f"train {len(X_train)}, validation {len(X_val)}, test {len(X_test)}")

### 3. Preprocessing without leakage
We standardize each feature (subtract the mean, divide by the standard deviation). The mean and standard deviation are computed **on the training set only** and then applied to all three sets: the validation and test sets must play the role of data we have never seen.

Some pixels (e.g. the image corners) are always 0 in the training set, so their standard deviation is 0: we replace it by 1 to avoid dividing by zero.

In [ ]:
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
std[std == 0] = 1.0

def to_tensors(X_part, y_part):
    X_t = torch.tensor((X_part - mean) / std, dtype=torch.float32)
    y_t = torch.tensor(y_part, dtype=torch.long)   # CrossEntropyLoss wants integer class labels
    return X_t, y_t

X_train_t, y_train_t = to_tensors(X_train, y_train)
X_val_t, y_val_t = to_tensors(X_val, y_val)
X_test_t, y_test_t = to_tensors(X_test, y_test)
print(X_train_t.shape, X_train_t.dtype, y_train_t.dtype)

### 4. Datasets and DataLoaders
When the data are already tensors, `TensorDataset` saves us from writing a custom `Dataset` class (notebook 07). Only the training loader shuffles; giving it its own random generator makes the order of the batches reproducible.

In [ ]:
train_ds = TensorDataset(X_train_t, y_train_t)
val_ds = TensorDataset(X_val_t, y_val_t)
test_ds = TensorDataset(X_test_t, y_test_t)

def make_train_loader(dataset, seed, batch_size=64):
    return DataLoader(dataset, batch_size=batch_size, shuffle=True,
                      generator=torch.Generator().manual_seed(seed))

train_loader = make_train_loader(train_ds, SEED)
val_loader = DataLoader(val_ds, batch_size=256)
test_loader = DataLoader(test_ds, batch_size=256)

xb, yb = next(iter(train_loader))
print("one batch:", xb.shape, yb.shape)

### 5. Models
A number such as "97% accuracy" means little on its own: we compare it with **baselines**.
- **Majority class:** always predict the most frequent class of the training set.
- **Linear model:** a single `nn.Linear(64, 10)` layer (multinomial logistic regression).
- **MLP:** two hidden layers with ReLU activations and optional dropout.

`nn.CrossEntropyLoss` takes the raw outputs of the last layer (*logits*) and applies the softmax internally: the model must **not** end with a softmax.

In [ ]:
n_features, n_classes = X_train.shape[1], len(np.unique(y_train))

def make_linear():
    return nn.Linear(n_features, n_classes)

def make_mlp(hidden=128, p_drop=0.0):
    return nn.Sequential(
        nn.Linear(n_features, hidden), nn.ReLU(), nn.Dropout(p_drop),
        nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(p_drop),
        nn.Linear(hidden, n_classes),
    )

print(make_mlp())

### 6. Training and evaluation functions
- `model.train()` and `model.eval()` switch layers such as dropout between their training and evaluation behaviour.
- Evaluation runs under `torch.no_grad()`: no graph is needed.
- `fit` keeps the weights of the epoch with the best **validation** accuracy: a simple form of *early stopping*.

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss, n = 0.0, 0
    for xb, yb in loader:
        logits = model(xb)
        loss = loss_fn(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(xb)
        n += len(xb)
    return total_loss / n


@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for xb, yb in loader:
        logits = model(xb)
        total_loss += loss_fn(logits, yb).item() * len(xb)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        n += len(xb)
    return total_loss / n, correct / n


def fit(model, train_loader, val_loader, epochs=50, lr=1e-3, weight_decay=0.0):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    best = {"val_acc": -1.0, "epoch": None, "state": None}
    for epoch in range(epochs):
        train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer)
        val_loss, val_acc = evaluate(model, val_loader, loss_fn)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        if val_acc > best["val_acc"]:
            best = {"val_acc": val_acc, "epoch": epoch, "state": copy.deepcopy(model.state_dict())}
    model.load_state_dict(best["state"])   # restore the best epoch
    return history, best["epoch"]

### 7. Train the linear model and the MLP

In [ ]:
set_seed(SEED)
linear = make_linear()
hist_linear, best_epoch_linear = fit(linear, make_train_loader(train_ds, SEED), val_loader)

set_seed(SEED)
mlp = make_mlp(hidden=128, p_drop=0.2)
hist_mlp, best_epoch_mlp = fit(mlp, make_train_loader(train_ds, SEED), val_loader)

print(f"linear: best validation accuracy {max(hist_linear['val_acc']):.3f} at epoch {best_epoch_linear}")
print(f"MLP:    best validation accuracy {max(hist_mlp['val_acc']):.3f} at epoch {best_epoch_mlp}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for name, hist in [("linear", hist_linear), ("MLP", hist_mlp)]:
    ax1.plot(hist["train_loss"], label=f"{name} train")
    ax1.plot(hist["val_loss"], "--", label=f"{name} validation")
    ax2.plot(hist["val_acc"], label=name)
ax1.set_xlabel("epoch")
ax1.set_ylabel("cross-entropy loss")
ax1.set_yscale("log")
ax1.legend()
ax2.set_xlabel("epoch")
ax2.set_ylabel("validation accuracy")
ax2.legend()
plt.tight_layout()
plt.show()

### 8. Final evaluation on the test set
All choices are made: we now evaluate each model **once** on the test set.

The test set is finite, so the accuracy has a statistical uncertainty. For $n$ independent test samples and accuracy $p$, the standard error is approximately $\sqrt{p(1-p)/n}$.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
majority_class = np.bincount(y_train).argmax()
results = {"majority class": float((y_test == majority_class).mean())}
for name, model in [("linear", linear), ("MLP", mlp)]:
    results[name] = evaluate(model, test_loader, loss_fn)[1]

n_test = len(y_test)
for name, acc in results.items():
    se = np.sqrt(acc * (1 - acc) / n_test)
    print(f"{name:>15}: test accuracy {acc:.3f} ± {se:.3f}")

A confusion matrix shows *which* classes are confused with each other.

In [ ]:
@torch.no_grad()
def predict(model, X_t):
    model.eval()
    return model(X_t).argmax(dim=1).numpy()

y_pred = predict(mlp, X_test_t)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("predicted class")
plt.ylabel("true class")
plt.title("MLP: confusion matrix on the test set")
plt.show()

wrong = np.flatnonzero(y_pred != y_test)
print(f"{len(wrong)} misclassified test images out of {n_test}")
fig, axes = plt.subplots(1, min(8, len(wrong)), figsize=(10, 2), squeeze=False)
for ax, i in zip(axes.flat, wrong[:8]):
    ax.imshow(X_test[i].reshape(8, 8), cmap="gray_r")
    ax.set_title(f"true {y_test[i]}, pred {y_pred[i]}", fontsize=8)
    ax.axis("off")
plt.show()

### 9. Save the model and the information needed to reproduce it
The `state_dict` contains the weights. To use them again we also need the architecture, the preprocessing (mean and standard deviation of the training set) and, to reproduce the training, the seed, hyperparameters and library versions.

In [ ]:
torch.save({"state_dict": mlp.state_dict(), "mean": mean, "std": std}, "digits_mlp.pt")

run_info = {
    "seed": SEED,
    "split": {"train": len(X_train), "validation": len(X_val), "test": len(X_test)},
    "model": {"type": "MLP", "hidden": 128, "p_drop": 0.2},
    "training": {"optimizer": "Adam", "lr": 1e-3, "epochs": 50, "batch_size": 64, "best_epoch": best_epoch_mlp},
    "test_accuracy": results["MLP"],
    "versions": versions,
}
with open("digits_mlp_run.json", "w") as f:
    json.dump(run_info, f, indent=2)

# Reload into a fresh model and check that we get the same predictions
checkpoint = torch.load("digits_mlp.pt", weights_only=False)
reloaded = make_mlp(hidden=128, p_drop=0.2)
reloaded.load_state_dict(checkpoint["state_dict"])
print("same predictions after reloading:", np.array_equal(predict(reloaded, X_test_t), y_pred))

## Part 2: Exercises & Debugging (90 mins)

### Lab 9.1: Overfitting and regularization
* **Task:**
  1. Keep only the first 10% of the training set (`X_train_t[:n_small]`, `y_train_t[:n_small]`) and build a new training loader.
  2. Train a large MLP (`hidden=512`, `p_drop=0.0`) for 200 epochs and plot training and validation loss. What do you observe?
  3. Repeat with `p_drop=0.3` and `weight_decay=1e-3`. Compare the best validation accuracies.

In [ ]:
# Your code here


### Lab 9.2: Is the MLP really better than the linear model?
One training run gives one number. Part of the difference between two models is due to chance (initialization, batch order).
* **Task:** Train the linear model and the MLP (`hidden=128`, `p_drop=0.2`) with 5 different seeds each (same data split). Report the mean and standard deviation of the best **validation** accuracy for both models.
* **Question:** Why do we use the validation set and not the test set for this comparison?

In [ ]:
# Your code here


### Lab 9.3 (optional): A different dataset
* **Task:** Run the same pipeline on `sklearn.datasets.load_breast_cancer()` (569 samples, 30 numerical features, 2 classes). What do you need to change? Compare the MLP with the majority-class baseline on the test set.

In [ ]:
# Your code here


### Debugging Focus: Common mistakes in classification pipelines
- **Softmax before `nn.CrossEntropyLoss`:** the loss already applies (log-)softmax. Adding `nn.Softmax` at the end of the model applies it twice: training becomes slow and unstable.
- **Wrong label type:** `nn.CrossEntropyLoss` expects class indices of type `torch.long` with shape `(batch,)`, not floats and not one-hot vectors.
- **Forgetting `model.eval()`:** dropout stays active during evaluation, and the accuracy changes from one evaluation to the next.
- **Data leakage:** computing the standardization statistics on the full dataset, or choosing hyperparameters on the test set. The code runs and the numbers look good, but they overestimate the performance on new data.